# Notebook 2: Self-Correcting Code Agent

**Cursor Features We're Building:** Bugbot (detect and fix bugs automatically), Inline Edit (modify existing code), Rules (custom instructions that shape behavior)

In Notebook 1 we built an agent that generates code and writes files. But it has no idea if the code actually works. In this notebook we add the ability to **execute code, detect errors, and fix them automatically** — the same loop that powers Cursor's Bugbot.

### What's New

| Concept | Why |
|---|---|
| `with_structured_output` | Force the LLM to return Pydantic-validated structured responses |
| Conditional retry loop | Route back to generator on error, with a bounded retry count |
| Reflection node | A separate LLM call that reviews code quality after execution succeeds |
| Dynamic rules injection | Inject custom coding rules into the system prompt at runtime |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("API Key loaded" if api_key else "API Key NOT found")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
)

response = llm.invoke("Say 'ready' if you can hear me.")
print(response.content)

## Structured Output with `with_structured_output`

In NB1 the LLM returned free-text. For a code agent we need **structured responses** — the code separated from the explanation. `with_structured_output` binds a Pydantic model to the LLM so every response is validated.

In [ ]:
from pydantic import BaseModel, Field


class CodeOutput(BaseModel):
    code: str = Field(description="The complete Python code to execute")
    explanation: str = Field(description="Brief explanation of what the code does")


structured_llm = llm.with_structured_output(CodeOutput)

result = structured_llm.invoke(
    "Write a Python function that checks if a number is prime"
)
print(f"Type: {type(result)}")
print(f"Explanation: {result.explanation}")
print(f"Code:\n{result.code}")

The LLM now always returns a `CodeOutput` object — no parsing needed. This is how Cursor separates the code it generates from its explanation text.

## Code Execution

To self-correct, the agent needs to **run** the code. We create a simple executor that captures stdout and stderr.

In [ ]:
import subprocess


def execute_python(code: str) -> dict:
    """Execute Python code in a subprocess and return stdout/stderr."""
    result = subprocess.run(
        ["python", "-c", code],
        capture_output=True,
        text=True,
        timeout=10,
    )
    return {
        "stdout": result.stdout,
        "stderr": result.stderr,
        "returncode": result.returncode,
    }


# Test with working code
out = execute_python("print('hello world')")
print("Working code:", out)

# Test with broken code
out = execute_python("print(1/0)")
print("Broken code:", out)

## Building the Self-Correcting Graph

The core idea: **Generate → Execute → if error, loop back to Generate with the error message.**

This is the Prompt Chaining pattern (sequential steps) combined with the Reflection pattern (evaluate output, retry if needed). We cap retries at 3 to avoid infinite loops (Exception Handling pattern).

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class AgentState(TypedDict):
    task: str
    code: str
    explanation: str
    execution_result: str
    error: str
    attempts: int
    max_attempts: int
    status: str  # "generating", "executing", "success", "failed"


print("State defined")

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage


def generate(state: AgentState) -> AgentState:
    """Generate code based on the task (and previous error if retrying)."""
    prompt = f"Write Python code for: {state['task']}"

    if state.get("error"):
        prompt += f"\n\nPrevious attempt failed with error:\n{state['error']}\n\nFix the code."

    result = structured_llm.invoke(prompt)

    return {
        "code": result.code,
        "explanation": result.explanation,
        "attempts": state.get("attempts", 0) + 1,
        "status": "executing",
        "error": "",
    }


def execute(state: AgentState) -> AgentState:
    """Execute the generated code and capture results."""
    result = execute_python(state["code"])

    if result["returncode"] == 0:
        return {
            "execution_result": result["stdout"],
            "error": "",
            "status": "success",
        }
    else:
        return {
            "execution_result": "",
            "error": result["stderr"],
            "status": "failed",
        }


def should_retry(state: AgentState) -> str:
    """Decide: success → end, error with retries left → regenerate, out of retries → end."""
    if state["status"] == "success":
        return "success"
    if state["attempts"] < state.get("max_attempts", 3):
        return "retry"
    return "give_up"


print("Nodes defined")

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("generate", generate)
graph.add_node("execute", execute)

graph.add_edge(START, "generate")
graph.add_edge("generate", "execute")
graph.add_conditional_edges(
    "execute",
    should_retry,
    {
        "success": END,
        "retry": "generate",
        "give_up": END,
    },
)

bugbot = graph.compile()
print("Self-correcting graph compiled")

In [ ]:
from IPython.display import Image, display

display(Image(bugbot.get_graph().draw_mermaid_png()))

Notice the loop: `execute` routes back to `generate` on failure. This is the conditional retry pattern — the same way Cursor's Bugbot detects an error and tries a fix.

## Test: Easy Task (should succeed first try)

In [ ]:
result = bugbot.invoke(
    {
        "task": "Print the first 10 Fibonacci numbers",
        "attempts": 0,
        "max_attempts": 3,
    }
)

print(f"Status: {result['status']}")
print(f"Attempts: {result['attempts']}")
print(f"Explanation: {result['explanation']}")
print(f"Output: {result['execution_result']}")
print(f"Code:\n{result['code']}")

## Test: Harder Task (may need retries)

Let's give it something more complex where the first attempt might fail.

In [ ]:
result = bugbot.invoke(
    {
        "task": """Create a function that:
1. Reads JSON from stdin  
2. Flattens nested dictionaries (e.g. {"a": {"b": 1}} -> {"a.b": 1})
3. Prints the flattened result

Test it with this input: {"user": {"name": "Alice", "address": {"city": "NYC", "zip": "10001"}}, "active": true}
Use json.loads on a hardcoded string to simulate stdin.""",
        "attempts": 0,
        "max_attempts": 3,
    }
)

print(f"Status: {result['status']} (after {result['attempts']} attempt(s))")
if result["error"]:
    print(f"Final error: {result['error'][:200]}")
print(f"Output: {result['execution_result']}")

## Watching the Retry Loop

Let's trace the graph step-by-step to see when it retries.

In [ ]:
inputs = {
    "task": "Use the 'statistics' module to compute mean, median, and stdev of [2, 4, 4, 4, 5, 5, 7, 9]. Print each on a separate line formatted as 'metric: value'.",
    "attempts": 0,
    "max_attempts": 3,
}

for step in bugbot.stream(inputs):
    node_name = list(step.keys())[0]
    state = step[node_name]

    if node_name == "generate":
        print(f"[generate] Attempt {state.get('attempts', '?')}")
        print(f"  Code preview: {state['code'][:80]}...")
    elif node_name == "execute":
        if state.get("error"):
            print(f"[execute] FAILED: {state['error'][:100]}")
        else:
            print(f"[execute] SUCCESS: {state['execution_result'][:100]}")
    print()

## Adding a Reviewer — The Reflection Pattern

Passing execution isn't enough. Code can run but still be bad (no type hints, poor naming, inefficient). We add a **Reviewer node** that evaluates code quality after successful execution.

This is the Reflection pattern — a separate LLM call that evaluates the output against quality criteria and can send it back for revision.

In [ ]:
class ReviewResult(BaseModel):
    approved: bool = Field(description="Whether the code meets quality standards")
    feedback: str = Field(
        description="Specific feedback for improvement if not approved"
    )


reviewer_llm = llm.with_structured_output(ReviewResult)

# Test the reviewer on some code
test_code = "x = [1,2,3]\nfor i in x:\n  print(i)"
review = reviewer_llm.invoke(
    f"Review this Python code for quality (type hints, naming, PEP 8, efficiency):\n\n{test_code}"
)
print(f"Approved: {review.approved}")
print(f"Feedback: {review.feedback}")

## Full Graph: Generate → Execute → Review

Now we wire it all together: generate code, execute it, if it passes execution send to reviewer, if reviewer rejects send back to generator with feedback.

In [ ]:
class FullAgentState(TypedDict):
    task: str
    rules: str  # custom coding rules
    code: str
    explanation: str
    execution_result: str
    error: str
    review_feedback: str
    attempts: int
    max_attempts: int
    status: str


def generate_v2(state: FullAgentState) -> FullAgentState:
    prompt = f"Write Python code for: {state['task']}"

    if state.get("rules"):
        prompt = f"Follow these rules:\n{state['rules']}\n\n{prompt}"

    if state.get("error"):
        prompt += f"\n\nPrevious attempt had an execution error:\n{state['error']}\n\nFix the code."
    elif state.get("review_feedback"):
        prompt += f"\n\nPrevious code worked but the reviewer said:\n{state['review_feedback']}\n\nImprove the code."

    result = structured_llm.invoke(prompt)
    return {
        "code": result.code,
        "explanation": result.explanation,
        "attempts": state.get("attempts", 0) + 1,
        "status": "executing",
        "error": "",
        "review_feedback": "",
    }


def execute_v2(state: FullAgentState) -> FullAgentState:
    result = execute_python(state["code"])
    if result["returncode"] == 0:
        return {"execution_result": result["stdout"], "error": "", "status": "executed"}
    else:
        return {
            "execution_result": "",
            "error": result["stderr"],
            "status": "exec_failed",
        }


def review(state: FullAgentState) -> FullAgentState:
    result = reviewer_llm.invoke(
        f"Review this Python code for quality (type hints, naming, PEP 8, efficiency, correctness):\n\n{state['code']}"
    )
    if result.approved:
        return {"status": "approved", "review_feedback": ""}
    else:
        return {"status": "review_failed", "review_feedback": result.feedback}


def route_after_execute(state: FullAgentState) -> str:
    if state["status"] == "executed":
        return "review"
    if state["attempts"] < state.get("max_attempts", 3):
        return "retry"
    return "give_up"


def route_after_review(state: FullAgentState) -> str:
    if state["status"] == "approved":
        return "done"
    if state["attempts"] < state.get("max_attempts", 3):
        return "revise"
    return "done"


print("Full agent nodes defined")

In [ ]:
graph = StateGraph(FullAgentState)

graph.add_node("generate", generate_v2)
graph.add_node("execute", execute_v2)
graph.add_node("review", review)

graph.add_edge(START, "generate")
graph.add_edge("generate", "execute")
graph.add_conditional_edges(
    "execute",
    route_after_execute,
    {
        "review": "review",
        "retry": "generate",
        "give_up": END,
    },
)
graph.add_conditional_edges(
    "review",
    route_after_review,
    {
        "done": END,
        "revise": "generate",
    },
)

full_agent = graph.compile()
print("Full agent compiled")

In [ ]:
display(Image(full_agent.get_graph().draw_mermaid_png()))

Two loops visible: execute can loop back on error, and review can loop back for quality improvements. Both bounded by `max_attempts`.

## Running the Full Agent

In [ ]:
result = full_agent.invoke(
    {
        "task": "Write a function to find all prime numbers up to n using the Sieve of Eratosthenes. Test it by printing primes up to 50.",
        "rules": "",
        "attempts": 0,
        "max_attempts": 3,
    }
)

print(f"Status: {result['status']} (after {result['attempts']} attempt(s))")
print(f"Output: {result['execution_result']}")
print(f"\nCode:\n{result['code']}")

## Tracing the Full Pipeline

Let's stream each step to see Generate -> Execute -> Review in action.

In [ ]:
for step in full_agent.stream(
    {
        "task": "Create a dataclass called 'Point' with x,y coordinates. Add methods for distance_to(other), midpoint(other), and __str__. Test with Point(3,4) and Point(0,0).",
        "rules": "",
        "attempts": 0,
        "max_attempts": 3,
    }
):
    node_name = list(step.keys())[0]
    state = step[node_name]

    if node_name == "generate":
        print(
            f"[generate] Attempt {state.get('attempts', '?')}: {state.get('explanation', '')[:100]}"
        )
    elif node_name == "execute":
        if state.get("error"):
            print(f"[execute] FAILED: {state['error'][:150]}")
        else:
            print(f"[execute] OK: {state['execution_result'][:150]}")
    elif node_name == "review":
        status = state.get("status", "")
        feedback = state.get("review_feedback", "")
        print(f"[review] {status}: {feedback[:150]}")
    print()

## Custom Rules — Like Cursor Rules

Cursor lets you define rules in `.cursorrules` files that shape how the AI writes code. We do the same by injecting custom rules into the generator's prompt at runtime. This is dynamic state injection — the rules are part of the graph state, not hardcoded.

In [ ]:
# Without rules
result_no_rules = full_agent.invoke(
    {
        "task": "Write a function to sort a list of dictionaries by a given key. Test with sample data.",
        "rules": "",
        "attempts": 0,
        "max_attempts": 3,
    }
)

print("=== Without Rules ===")
print(result_no_rules["code"])

In [ ]:
# With strict rules
STRICT_RULES = """- ALL functions must have type hints on parameters and return type
- ALL functions must have a Google-style docstring
- Use list comprehensions instead of loops where possible
- Add if __name__ == '__main__' guard for test code
- Variable names must be descriptive (no single letters except loop counters)"""

result_with_rules = full_agent.invoke(
    {
        "task": "Write a function to sort a list of dictionaries by a given key. Test with sample data.",
        "rules": STRICT_RULES,
        "attempts": 0,
        "max_attempts": 3,
    }
)

print("=== With Rules ===")
print(result_with_rules["code"])

## Inline Edit — Modify Existing Code

Cursor's Inline Edit (Cmd+K) takes existing code and modifies it based on instructions. We build this by passing existing code + edit instructions to the generator.

In [ ]:
existing_code = """
def greet(name):
    print("Hello " + name)

greet("World")
"""

result = full_agent.invoke(
    {
        "task": f"""Modify this existing code:
```python
{existing_code}
```

Changes requested:
- Add type hints
- Add a docstring
- Support an optional greeting parameter (default "Hello")
- Return the string instead of printing it
- Add tests that verify the output""",
        "rules": "",
        "attempts": 0,
        "max_attempts": 3,
    }
)

print(f"Status: {result['status']} (attempts: {result['attempts']})")
print(f"Output: {result['execution_result']}")
print(f"\nModified code:\n{result['code']}")

## Combining Rules + Inline Edit

This is where it gets powerful — edit existing code while enforcing custom rules.

In [ ]:
legacy_code = """
import csv

def read_data(file):
    f = open(file)
    r = csv.reader(f)
    data = []
    for row in r:
        data.append(row)
    f.close()
    return data

d = read_data("test.csv")
print(d)
"""

MODERNIZE_RULES = """- Use context managers (with statement) for file handling
- Use pathlib.Path instead of string paths
- Use list comprehensions where appropriate
- Add proper error messages
- Use type hints everywhere"""

result = full_agent.invoke(
    {
        "task": f"""Modernize this legacy code:
```python
{legacy_code}
```
Rewrite it following modern Python best practices. Create a small test CSV inline using io.StringIO for testing.""",
        "rules": MODERNIZE_RULES,
        "attempts": 0,
        "max_attempts": 3,
    }
)

print(f"Status: {result['status']} (attempts: {result['attempts']})")
print(f"Output: {result['execution_result']}")
print(f"\nModernized code:\n{result['code']}")

## Recap

| What we built | Cursor equivalent |
|---|---|
| `with_structured_output` for code/explanation | Cursor separating code from explanation in Chat |
| Generate -> Execute retry loop | Bugbot detecting and auto-fixing errors |
| Reviewer node with quality checks | Cursor's code review suggestions |
| Custom rules in state | Cursor Rules (`.cursorrules` files) |
| Inline edit (modify existing code) | Cmd+K inline edit |

### Design Patterns Used

- **Prompt Chaining** -- Generate -> Execute -> Review as sequential pipeline
- **Reflection** -- Reviewer evaluates output quality, can reject and retry
- **Exception Handling** -- Bounded retries, categorized failures (exec error vs review rejection)

### Next Notebook

The agent works on single tasks. In Notebook 3 we build the full production agent: multiple specialized agents, codebase search, human review before applying changes, parallel code generation, and shell command execution.